# Day 3 - Pinecone
## Semantic E-Commerce Product Search

Pinecone is a fully managed, purpose-built vector database. Unlike Day 1 and Day 2 where vector search was an extension or add-on to an existing database, Pinecone exists solely to store and search vectors. There is no relational layer, no document model and no infrastructure to manage - just an API.

**When would you reach for this?**
- You want vector search without managing any infrastructure
- Your use case is pure similarity search with metadata filtering
- You need to scale to hundreds of millions of vectors without operational overhead

**The use case:** A semantic product search engine for an electronics store where users describe what they are looking for in natural language - "a laptop for video editing under $1500" or "noise canceling headphones for travel" - and get relevant products in return.

## 1. Setup

### Prerequisites

- A Pinecone account with an API key
- Ollama running locally with the `all-minilm` model pulled
- Python 3.12 with a virtual environment

### Set Environment Variable

Set your Pinecone API Key as an environment variable before running the notebook:

```shell
export PINECONE_API_KEY="your-api-key"
```

### Install Python Dependencies

In [1]:
%pip install ollama==0.6.2 \
             pandas==3.0.3 \
             pinecone==6.0.2 \
             tqdm==4.67.1 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ollama
import os
import random
import pandas as pd
import time

from pinecone import Pinecone, ServerlessSpec
from tqdm.notebook import tqdm

### Configuration

`NUM_PRODUCTS` controls the size of the generated dataset. 200 is the recommended default for this tutorial.

> **Note:** Embedding generation runs locally via Ollama and is single-threaded. At 200 products the notebook runs comfortably. Larger values will work but will take proportionally longer.

In [3]:
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
INDEX_NAME       = "electronics"
CLOUD            = "aws"
REGION           = "us-east-1"
LLM_EMBEDDING    = "all-minilm"
NUM_PRODUCTS     = 200
RANDOM_SEED      = 42

### Verify Ollama is Running

In [4]:
ollama_ready = False

try:
    models = ollama.list()
    model_names = [m.model for m in models.models]
    print("Available models:", model_names)
    assert any(LLM_EMBEDDING in m for m in model_names)
    print(f"Model '{LLM_EMBEDDING}' is ready.")
    ollama_ready = True
except ConnectionError:
    print("ERROR: Ollama is not running. Start it with: ollama serve")
except AssertionError:
    print(f"ERROR: Model not found. Run: ollama pull {LLM_EMBEDDING}")

Available models: ['llama3:latest', 'all-minilm:latest']
Model 'all-minilm' is ready.


In [5]:
assert ollama_ready, "Please fix the Ollama issue above before continuing."

### Determine Embedding Dimensions

We determine the embedding dimensions dynamically so the Pinecone index definition stays in sync with whichever embedding model is in use.

In [6]:
def get_embedding(text: str) -> list:
    response = ollama.embeddings(model = LLM_EMBEDDING, prompt = text)
    return response["embedding"]

test_embedding = get_embedding("noise canceling headphones for travel")
EMBEDDING_DIMS = len(test_embedding)
print(f"Embedding dimensions: {EMBEDDING_DIMS}")

Embedding dimensions: 384


### Connect to Pinecone

In [7]:
pc = Pinecone(api_key = PINECONE_API_KEY)
print("Connected to Pinecone.")

Connected to Pinecone.


## 2. The Dataset

We generate electronics product listings programmatically from pools of categories, brands, attributes and description templates. Each product description is a short prose summary of the product - this is what we embed and search over.

In Pinecone, structured fields such as category, brand and price are stored as **metadata** on each vector record rather than in a separate table or document. Metadata is used for filtering at query time.

In [8]:
random.seed(RANDOM_SEED)

CATEGORIES = [
    "Laptop", "Smartphone", "Tablet", "Headphones", "Camera",
    "Monitor", "Keyboard", "Mouse", "Speaker", "Smartwatch",
    "TV", "Gaming Console", "Drone", "Projector", "E-Reader",
]

BRANDS = [
    "TechPro", "NovaByte", "PixelEdge", "SoundWave", "VisionCore",
    "SwiftTech", "PeakGear", "ClearView", "BrightLink", "ZenTech",
    "AlphaCore", "PulseAudio", "NexaDisplay", "CloudSync", "AeroSense",
]

PRICE_BANDS = {
    "Laptop":          (599,  2499),
    "Smartphone":      (299,  1399),
    "Tablet":          (199,  1099),
    "Headphones":      (49,   499),
    "Camera":          (399,  2999),
    "Monitor":         (149,  1299),
    "Keyboard":        (29,   299),
    "Mouse":           (19,   199),
    "Speaker":         (49,   599),
    "Smartwatch":      (99,   799),
    "TV":              (299,  2999),
    "Gaming Console":  (299,  699),
    "Drone":           (199,  1999),
    "Projector":       (299,  1999),
    "E-Reader":        (79,   349),
}

PERFORMANCE = ["entry-level", "mid-range", "high-performance", "professional-grade", "flagship"]

USE_CASES = [
    "everyday use", "creative professionals", "students", "remote workers",
    "gamers", "travelers", "content creators", "home entertainment",
    "photography enthusiasts", "business users", "audiophiles",
]

FEATURES = {
    "Laptop":         ["long battery life", "lightweight design", "fast SSD storage",
                       "dedicated GPU", "high-resolution display", "backlit keyboard"],
    "Smartphone":     ["triple camera system", "all-day battery", "5G connectivity",
                       "fast charging", "AMOLED display", "water resistance"],
    "Tablet":         ["stylus support", "portable form factor", "sharp display",
                       "long battery life", "detachable keyboard", "cellular connectivity"],
    "Headphones":     ["active noise cancellation", "wireless connectivity", "premium sound quality",
                       "long battery life", "foldable design", "built-in microphone"],
    "Camera":         ["full-frame sensor", "4K video recording", "optical image stabilization",
                       "fast autofocus", "weather sealing", "interchangeable lenses"],
    "Monitor":        ["4K resolution", "high refresh rate", "IPS panel", "ultra-wide format",
                       "USB-C connectivity", "HDR support"],
    "Keyboard":       ["mechanical switches", "RGB backlighting", "wireless connectivity",
                       "compact layout", "programmable keys", "ergonomic design"],
    "Mouse":          ["high DPI sensor", "ergonomic shape", "wireless connectivity",
                       "programmable buttons", "long battery life", "silent clicks"],
    "Speaker":        ["360 degree sound", "waterproof design", "deep bass",
                       "multi-room audio", "voice assistant support", "long battery life"],
    "Smartwatch":     ["health monitoring", "GPS tracking", "sleep tracking",
                       "water resistance", "long battery life", "always-on display"],
    "TV":             ["4K OLED display", "HDR support", "smart TV platform",
                       "Dolby Atmos audio", "thin bezel design", "gaming mode"],
    "Gaming Console": ["4K gaming", "ray tracing", "fast SSD storage",
                       "backward compatibility", "online multiplayer", "VR support"],
    "Drone":          ["4K camera", "obstacle avoidance", "long flight time",
                       "GPS stabilization", "foldable design", "live video feed"],
    "Projector":      ["4K resolution", "short throw lens", "high brightness",
                       "built-in speakers", "wireless connectivity", "long lamp life"],
    "E-Reader":       ["glare-free display", "waterproof design", "weeks of battery life",
                       "adjustable warm light", "lightweight build", "large storage"],
}

DESCRIPTION_TEMPLATES = [
    "A {performance} {category} designed for {use_case}, featuring {feature1} and {feature2}.",
    "Built for {use_case}, this {performance} {category} delivers {feature1} alongside {feature2}.",
    "The {brand} {category} is a {performance} option for {use_case}, offering {feature1} and {feature2}.",
    "Ideal for {use_case}, this {category} combines {feature1} with {feature2} in a {performance} package.",
    "A {performance} {category} with {feature1} and {feature2}, perfect for {use_case}.",
]

def generate_product() -> dict:
    category    = random.choice(CATEGORIES)
    brand       = random.choice(BRANDS)
    performance = random.choice(PERFORMANCE)
    use_case    = random.choice(USE_CASES)
    features    = random.sample(FEATURES[category], 2)
    price_min, price_max = PRICE_BANDS[category]
    price       = round(random.randint(price_min, price_max) / 10) * 10

    description = random.choice(DESCRIPTION_TEMPLATES).format(
        category    = category,
        brand       = brand,
        performance = performance,
        use_case    = use_case,
        feature1    = features[0],
        feature2    = features[1],
    )

    name = f"{brand} {category} {random.randint(100, 999)}"

    return {
        "name":        name,
        "category":    category,
        "brand":       brand,
        "price":       price,
        "performance": performance,
        "use_case":    use_case,
        "description": description,
    }

products = [generate_product() for _ in range(NUM_PRODUCTS)]
df = pd.DataFrame(products)
print(f"Generated {len(df)} products")
df.head()

Generated 200 products


,name,category,brand,price,performance,use_case,description
0,NovaByte TV 792,TV,NovaByte,870,entry-level,gamers,"A entry-level TV designed for gamers, featurin..."
1,AeroSense Gaming Console 195,Gaming Console,AeroSense,320,flagship,creative professionals,A flagship Gaming Console designed for creativ...
2,SoundWave Headphones 529,Headphones,SoundWave,150,flagship,business users,A flagship Headphones with active noise cancel...
3,ClearView Headphones 448,Headphones,ClearView,410,flagship,gamers,"Ideal for gamers, this Headphones combines act..."
4,PixelEdge Camera 467,Camera,PixelEdge,1960,mid-range,travelers,"A mid-range Camera designed for travelers, fea..."


## 3. Create the Pinecone Index

Pinecone organises vectors in **indexes**. Each index stores vectors of a fixed dimension alongside optional metadata. We create a serverless index on AWS using the embedding dimensions determined earlier.

If the index already exists from a previous run, we delete it and recreate it for a clean start.

In [9]:
existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME in existing_indexes:
    print(f"Deleting existing index '{INDEX_NAME}'...")
    pc.delete_index(INDEX_NAME)
    print("Deleted.")

print(f"Creating index '{INDEX_NAME}'...")
pc.create_index(
    name      = INDEX_NAME,
    dimension = EMBEDDING_DIMS,
    metric    = "cosine",
    spec      = ServerlessSpec(cloud = CLOUD, region = REGION)
)

print(f"Index '{INDEX_NAME}' created.")

Creating index 'electronics'...
Index 'electronics' created.


### Wait for the Index to be Ready

In [10]:
print("Waiting for index to be ready...")
while True:
    status = pc.describe_index(INDEX_NAME).status
    if status.get("ready"):
        print(f"Index '{INDEX_NAME}' is ready.")
        break
    print(f"  Status: {status} - waiting...")
    time.sleep(5)

index = pc.Index(INDEX_NAME)

Waiting for index to be ready...
Index 'electronics' is ready.


## 4. Generate Embeddings and Load Data

In Pinecone each record consists of:
- `id` - a unique string identifier
- `values` - the embedding vector
- `metadata` - a dictionary of structured fields for filtering

We upsert records in batches for efficiency. Pinecone's `upsert` operation inserts new records and updates existing ones.

In [11]:
BATCH_SIZE = 50

records = []
for i, product in enumerate(tqdm(products, desc = "Generating embeddings")):
    embedding = get_embedding(product["description"])
    records.append({
        "id":     str(i),
        "values": embedding,
        "metadata": {
            "name":        product["name"],
            "category":    product["category"],
            "brand":       product["brand"],
            "price":       product["price"],
            "performance": product["performance"],
            "use_case":    product["use_case"],
            "description": product["description"],
        }
    })

print("\nUpserting records into Pinecone...")
for i in range(0, len(records), BATCH_SIZE):
    batch = records[i:i + BATCH_SIZE]
    index.upsert(vectors = batch)

print(f"Upserted {len(records)} records.")

Generating embeddings:   0%|          | 0/200 [00:00<?, ?it/s]


Upserting records into Pinecone...
Upserted 200 records.


### Verify the Index

In [12]:
stats = index.describe_index_stats()
print(f"Total vectors in index: {stats['total_vector_count']}")
print(f"Dimensions: {stats['dimension']}")

Total vectors in index: 200
Dimensions: 384


## 5. Semantic Search

We embed the user's query and pass it to Pinecone's `query` method. Pinecone returns the most similar vectors ranked by cosine similarity, along with their metadata.

In [13]:
def search_products(query: str, top_k: int = 5):
    query_embedding = get_embedding(query)

    results = index.query(
        vector = query_embedding,
        top_k = top_k,
        include_metadata = True
    )

    print(f"\nQuery: '{query}'\n")
    for match in results["matches"]:
        m = match["metadata"]
        print(f"  {m['name']} ({m['category']})")
        print(f"  ${m['price']:,.0f} | {m['performance']} | Score: {match['score']:.3f}")
        print(f"  {m['description']}")
        print()

In [14]:
search_products("noise canceling headphones for travel")


Query: 'noise canceling headphones for travel'

  SoundWave Headphones 529 (Headphones)
  $150 | flagship | Score: 0.649
  A flagship Headphones with active noise cancellation and foldable design, perfect for business users.

  VisionCore Headphones 803 (Headphones)
  $460 | entry-level | Score: 0.621
  A entry-level Headphones with active noise cancellation and foldable design, perfect for content creators.

  VisionCore Headphones 777 (Headphones)
  $180 | high-performance | Score: 0.616
  A high-performance Headphones with active noise cancellation and wireless connectivity, perfect for home entertainment.

  ZenTech Headphones 678 (Headphones)
  $260 | flagship | Score: 0.609
  A flagship Headphones with foldable design and active noise cancellation, perfect for everyday use.

  ClearView Headphones 448 (Headphones)
  $410 | flagship | Score: 0.566
  Ideal for gamers, this Headphones combines active noise cancellation with wireless connectivity in a flagship package.



In [15]:
search_products("a laptop for video editing under $1500")


Query: 'a laptop for video editing under $1500'

  BrightLink Laptop 876 (Laptop)
  $1,600 | mid-range | Score: 0.623
  A mid-range Laptop with lightweight design and dedicated GPU, perfect for creative professionals.

  VisionCore Laptop 238 (Laptop)
  $2,420 | professional-grade | Score: 0.568
  A professional-grade Laptop with long battery life and lightweight design, perfect for creative professionals.

  NovaByte Laptop 710 (Laptop)
  $680 | high-performance | Score: 0.534
  A high-performance Laptop designed for students, featuring high-resolution display and fast SSD storage.

  NexaDisplay Laptop 997 (Laptop)
  $1,030 | high-performance | Score: 0.521
  A high-performance Laptop with fast SSD storage and long battery life, perfect for content creators.

  ZenTech Laptop 182 (Laptop)
  $720 | professional-grade | Score: 0.639
  A professional-grade Laptop with high-resolution display and lightweight design, perfect for photography enthusiasts.



In [16]:
search_products("a camera for wildlife photography")


Query: 'a camera for wildlife photography'

  AlphaCore Camera 552 (Camera)
  $1,050 | entry-level | Score: 0.536
  A entry-level Camera designed for home entertainment, featuring optical image stabilization and weather sealing.

  PixelEdge Camera 467 (Camera)
  $1,960 | mid-range | Score: 0.512
  A mid-range Camera designed for travelers, featuring full-frame sensor and interchangeable lenses.

  PeakGear Drone 325 (Drone)
  $1,840 | professional-grade | Score: 0.510
  A professional-grade Drone with live video feed and foldable design, perfect for photography enthusiasts.

  BrightLink Camera 745 (Camera)
  $900 | mid-range | Score: 0.493
  A mid-range Camera designed for gamers, featuring fast autofocus and interchangeable lenses.

  ZenTech Camera 999 (Camera)
  $1,620 | flagship | Score: 0.486
  A flagship Camera designed for everyday use, featuring interchangeable lenses and weather sealing.



## 6. Filtered Search

Pinecone supports metadata filtering at query time. Filters are applied before the vector search, so only matching records are considered as candidates. This is the same pre-filtering approach we saw in MongoDB Atlas.

Filters use a MongoDB-style query syntax.

In [17]:
def search_products_filtered(
    query:       str,
    category:    str   = None,
    max_price:   float = None,
    performance: str   = None,
    top_k:       int   = 5
):
    query_embedding = get_embedding(query)

    filter_doc = {}
    if category:
        filter_doc["category"]    = {"$eq": category}
    if max_price:
        filter_doc["price"]       = {"$lte": max_price}
    if performance:
        filter_doc["performance"] = {"$eq": performance}

    results = index.query(
        vector           = query_embedding,
        top_k            = top_k,
        include_metadata = True,
        filter           = filter_doc if filter_doc else None
    )

    label = f"query = '{query}'"
    if category:    label += f", category = '{category}'"
    if max_price:   label += f", max_price = ${max_price:,.0f}"
    if performance: label += f", performance = '{performance}'"
    print(f"\n{label}\n")

    for match in results["matches"]:
        m = match["metadata"]
        print(f"  {m['name']} ({m['category']})")
        print(f"  ${m['price']:,.0f} | {m['performance']} | Score: {match['score']:.3f}")
        print(f"  {m['description']}")
        print()

# Headphones under $200
search_products_filtered(
    "wireless headphones with great sound",
    category  = "Headphones",
    max_price = 200
)


query = 'wireless headphones with great sound', category = 'Headphones', max_price = $200

  VisionCore Headphones 777 (Headphones)
  $180 | high-performance | Score: 0.721
  A high-performance Headphones with active noise cancellation and wireless connectivity, perfect for home entertainment.

  CloudSync Headphones 979 (Headphones)
  $50 | mid-range | Score: 0.565
  The CloudSync Headphones is a mid-range option for travelers, offering wireless connectivity and foldable design.

  CloudSync Headphones 179 (Headphones)
  $190 | high-performance | Score: 0.571
  Ideal for photography enthusiasts, this Headphones combines long battery life with wireless connectivity in a high-performance package.

  NexaDisplay Headphones 609 (Headphones)
  $110 | mid-range | Score: 0.584
  A mid-range Headphones designed for gamers, featuring premium sound quality and built-in microphone.

  SoundWave Headphones 529 (Headphones)
  $150 | flagship | Score: 0.601
  A flagship Headphones with active nois

In [18]:
# High-performance laptops for creative professionals
search_products_filtered(
    "laptop for video editing and creative work",
    category    = "Laptop",
    performance = "high-performance"
)


query = 'laptop for video editing and creative work', category = 'Laptop', performance = 'high-performance'

  VisionCore Laptop 576 (Laptop)
  $1,840 | high-performance | Score: 0.523
  The VisionCore Laptop is a high-performance option for remote workers, offering dedicated GPU and high-resolution display.

  PulseAudio Laptop 201 (Laptop)
  $1,080 | high-performance | Score: 0.495
  The PulseAudio Laptop is a high-performance option for remote workers, offering dedicated GPU and backlit keyboard.

  NovaByte Laptop 710 (Laptop)
  $680 | high-performance | Score: 0.561
  A high-performance Laptop designed for students, featuring high-resolution display and fast SSD storage.

  NexaDisplay Laptop 997 (Laptop)
  $1,030 | high-performance | Score: 0.583
  A high-performance Laptop with fast SSD storage and long battery life, perfect for content creators.



In [19]:
# Any category, strict price cap
search_products_filtered(
    "portable device for travel",
    max_price = 300
)


query = 'portable device for travel', max_price = $300

  SwiftTech Tablet 259 (Tablet)
  $280 | high-performance | Score: 0.580
  A high-performance Tablet designed for travelers, featuring detachable keyboard and cellular connectivity.

  NovaByte E-Reader 820 (E-Reader)
  $300 | mid-range | Score: 0.529
  A mid-range E-Reader with weeks of battery life and waterproof design, perfect for travelers.

  SoundWave E-Reader 420 (E-Reader)
  $290 | professional-grade | Score: 0.517
  A professional-grade E-Reader designed for travelers, featuring adjustable warm light and large storage.

  SoundWave Tablet 517 (Tablet)
  $260 | professional-grade | Score: 0.427
  Ideal for students, this Tablet combines cellular connectivity with portable form factor in a professional-grade package.

  CloudSync Keyboard 622 (Keyboard)
  $120 | high-performance | Score: 0.396
  A high-performance Keyboard designed for travelers, featuring RGB backlighting and wireless connectivity.



## Cleanup

In [20]:
# Optional - delete the index when done
pc.delete_index(INDEX_NAME)
print(f"Index '{INDEX_NAME}' deleted.")

Index 'electronics' deleted.
